In [1]:
# !pip -q install  fusion_solar_py pvlib retry-requests openmeteo_requests requests-cache python-telegram-bot

In [2]:
# !pip -q install git+https://github.com/EmaR97/EnergyManagementRL.git@testing-10

In [3]:
# from kaggle_secrets import UserSecretsClient

# user_secrets = UserSecretsClient()
# FUSION_SOLAR_CLIENT_PASSWORD = user_secrets.get_secret("FUSION_SOLAR_CLIENT_PASSWORD")
# FUSION_SOLAR_CLIENT_USERNAME = user_secrets.get_secret("FUSION_SOLAR_CLIENT_USERNAME")
# LAT = float(user_secrets.get_secret("LAT"))
# LON = float(user_secrets.get_secret("LON"))
# TOKEN =user_secrets.get_secret("TELEGRAM_BOT_TOKEN")
# ADMIN_ID =int(user_secrets.get_secret("TELEGRAM_ID"))

In [4]:
import os

from dotenv import load_dotenv

load_dotenv()
FUSION_SOLAR_CLIENT_PASSWORD = os.environ.get("FUSION_SOLAR_CLIENT_PASSWORD")
FUSION_SOLAR_CLIENT_USERNAME = os.environ.get("FUSION_SOLAR_CLIENT_USERNAME")
LAT = float(os.environ.get("LAT"))
LON = float(os.environ.get("LON"))
TOKEN = os.environ.get("TELEGRAM_BOT_TOKEN")
ADMIN_ID = int(os.environ.get("TELEGRAM_ID"))

In [5]:
import logging

# Create a logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Create a file handler for logging to a file
file_handler = logging.FileHandler('.log')
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

# Create a console handler for logging to the screen
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.WARNING)
console_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

# Add both handlers to the logger

for handler in logger.handlers[:]:
    logger.removeHandler(handler)
logger.addHandler(file_handler)
logger.addHandler(console_handler)

In [6]:
from energymanagementrl.production_forecast import *

panel_model = PanelModel(pdc0=0.42, temp_model_a=-3.56, temp_model_b=-0.075, delta_t=3, gamma_pdc=-0.004)
num_panels = 14
arrays = [ArrayConfig(name='sud_east', panel_model=panel_model, num_panels=num_panels, tilt_angle=25, azimuth=110),
          ArrayConfig(name='nord_west', panel_model=panel_model, num_panels=num_panels, tilt_angle=18, azimuth=290)]

_plant_config = PlantConfig(
    latitude=LAT, longitude=LON, timezone='Europe/Rome', inverter_pdc0=6, arrays=arrays
)
_production_forecaster = EnergyPredictionSystem(plant_config=_plant_config, open_meteo_client=OpenMeteoClient())

In [7]:
from energymanagementrl.fusion_solar_connector import *

_client = FusionSolarClientParsed(FUSION_SOLAR_CLIENT_USERNAME, FUSION_SOLAR_CLIENT_PASSWORD,
                                  huawei_subdomain="uni004eu5")
periodic_task = PeriodicTask(_client.keep_alive)
periodic_task.start()
_plant_id = _client.get_plant_ids()[0]
battery_id = _client.get_battery_ids(_plant_id)[0]

Periodic task started.


In [8]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

# Create a simple dummy environment used only to define the model struct, as the model will be used in inference
env = gym.Env()
env.action_space = spaces.Discrete(2)  # Two possible actions: 0 or 1
env.observation_space = spaces.Box(low=0, high=1000, shape=(55,), dtype=np.float64)  # 55 state variables

In [9]:
from energymanagementrl.rl import load_model_with_weights

# Load the DQN policy
_model = load_model_with_weights(
    env,
    '../data/trained_models/models/dqn_1.0_0.06_0.06_0.02_1000_l_2.policy_weights.pth',
    # 'dqn.2024_12_31_11_44_29.latest'
)

In [10]:
from energymanagementrl.rl.real_system_interaction import EnergyManagementSystem

system = EnergyManagementSystem(
    client=_client,
    plant_id=_plant_id,
    battery_id=battery_id,
    production_forecaster=_production_forecaster,
    model=_model,
)
system.set_active(True)

In [11]:
import threading

control_thread = threading.Thread(target=system.control_loop, daemon=True)
control_thread.start()

In [12]:
from energymanagementrl.interface import TelegramBot

bot = TelegramBot(system=system, token=TOKEN, allowed_users=[ADMIN_ID])

In [ ]:
import asyncio
import nest_asyncio

# Apply nest_asyncio to avoid event loop conflicts
nest_asyncio.apply()

# Start bot using the existing event loop
loop = asyncio.get_event_loop()
loop.create_task(bot.run())

2025-02-16 12:50:04,674 - WARNING - Missing values in history data. Filling with mean.
2025-02-16 12:50:15,803 - ERROR - Execution control failed for user 5541452419: ErrorCode.BATTERY_WORK_MODE
2025-02-16 12:51:48,687 - WARNING - Missing values in history data. Filling with mean.
2025-02-16 12:51:49,756 - ERROR - Execution control failed for user 5541452419: ErrorCode.BATTERY_WORK_MODE
2025-02-16 12:52:08,556 - WARNING - Missing values in history data. Filling with mean.
2025-02-16 12:52:19,555 - ERROR - Execution control failed for user 5541452419: ErrorCode.BATTERY_WORK_MODE
2025-02-16 12:52:28,256 - WARNING - Missing values in history data. Filling with mean.
2025-02-16 12:52:29,384 - ERROR - Execution control failed for user 5541452419: ErrorCode.BATTERY_WORK_MODE
2025-02-16 12:53:15,354 - WARNING - Battery Mode: FULLY_FEED_TO_GRID
2025-02-16 12:53:15,355 - ERROR - No error handlers are registered, logging exception.
Traceback (most recent call last):
  File "/home/emanuele/IdeaPr